## 0. Diagnostic: Cek Kesegaran Index ChromaDB

**Jalankan cell ini SEBELUM cell lain di bawah**, untuk memastikan `chroma_db/` yang di-load memang berasal dari `resep_5000_clean.parquet` versi TERBARU (setelah semua fix parser di notebook 01), bukan index lama yang masih kotor.

In [1]:
import chromadb

_client_check = chromadb.PersistentClient(path="chroma_db")
_col_check = _client_check.get_collection("resep_masakan")
print("Jumlah dokumen ter-index:", _col_check.count())

# cari beberapa resep yang KITA TAHU harus ada & bahannya harus bersih
# (kalau title ini tidak ketemu / bahannya masih ada 'gr'/'btr' dsb, index-nya STALE)
check_titles = ["Ayam Goreng Telur", "Nugget ayam wortel", "Garang Asem Ayam Telur"]

sample = _col_check.get(limit=5000, include=["metadatas"])
titles_in_index = {m["title"] for m in sample["metadatas"]}

for t in check_titles:
    print(f"{t!r} ada di index?", t in titles_in_index)

# cek kontaminasi unit di ingredient_names (harusnya 0 kalau sudah versi terbaru)
import json as _json

contaminated = 0
for m in sample["metadatas"]:
    names = _json.loads(m["ingredient_names"])
    if any(n.startswith(("gr ", "btr ", "bh ", "ml ")) for n in names):
        contaminated += 1
print(
    "Jumlah resep dgn nama bahan masih kecemar satuan (gr/btr/bh/ml di depan):",
    contaminated,
)
print(
    "-> Kalau angka ini TIDAK 0, index kamu STALE. Rebuild dari notebook 02 (uncomment delete_collection, lalu jalankan ulang dari load parquet)."
)

Jumlah dokumen ter-index: 5000
'Ayam Goreng Telur' ada di index? True
'Nugget ayam wortel' ada di index? True
'Garang Asem Ayam Telur' ada di index? True
Jumlah resep dgn nama bahan masih kecemar satuan (gr/btr/bh/ml di depan): 0
-> Kalau angka ini TIDAK 0, index kamu STALE. Rebuild dari notebook 02 (uncomment delete_collection, lalu jalankan ulang dari load parquet).


# Reranking: Skor SBERT + Skor Efisiensi Bahan

Lanjutan dari `02_embedding_indexing.ipynb`.

Ide dasar: retrieval SBERT murni bagus untuk menangkap kemiripan semantik umum ("masakan telur"), tapi tidak menjamin kelengkapan bahan sesuai input user. Di sini kita:
1. Retrieve kandidat lebih banyak dari SBERT/ChromaDB (misal top 30), bukan langsung top 5
2. Hitung **skor efisiensi bahan** = seberapa cocok/lengkap bahan resep dengan bahan yang dimiliki user
3. Gabungkan skor SBERT (dinormalisasi) + skor efisiensi bahan jadi skor akhir
4. Urutkan ulang (rerank), ambil top-N final untuk ditampilkan ke user

In [2]:
import re
import html
import json
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb

c:\Users\Mayceren\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load Model & Koneksi ke ChromaDB

Pastikan `chroma_db/` hasil dari notebook 02 ada di direktori yang sama.

In [3]:
MODEL_NAME = "LazarusNLP/all-indo-e5-small-v4"
CHROMA_DIR = "chroma_db"
COLLECTION_NAME = "resep_masakan"

model = SentenceTransformer(MODEL_NAME)
client = chromadb.PersistentClient(path=CHROMA_DIR)
collection = client.get_collection(COLLECTION_NAME)

print("Jumlah dokumen di collection:", collection.count())


def add_query_prefix(text: str) -> str:
    return f"query: {text}"


def add_passage_prefix(text: str) -> str:
    return f"passage: {text}"

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9950.37it/s]


Jumlah dokumen di collection: 5000


## 2. Parser Bahan untuk Input User

Sama seperti parser di `01_preprocessing.ipynb` (disalin ke sini supaya notebook ini bisa jalan standalone). Kalau nanti diintegrasikan ke aplikasi, sebaiknya fungsi ini dipindah ke satu modul `.py` bersama supaya tidak duplikat.

In [4]:
UNITS = [
    "sendok makan",
    "sendok teh",
    "sdm",
    "sdt",
    "gram",
    "gr",
    "kg",
    "ons",
    "ml",
    "mili",
    "liter",
    "ltr",
    "l",
    "butir",
    "btr",
    "buah",
    "bh",
    "siung",
    "lembar",
    "lbr",
    "batang",
    "btg",
    "ikat",
    "ruas",
    "cm",
    "potong",
    "ptg",
    "biji",
    "bj",
    "ekor",
    "ekr",
    "genggam",
    "bungkus",
    "bks",
    "sachet",
    "sct",
    "papan",
    "buku",
    "helai",
    "hlai",
    "gelas",
    "gls",
    "cup",
    "mangkuk",
    "piring",
    "sisir",
    "kotak",
    "ktk",
    "kaleng",
    "klg",
    "botol",
    "btl",
]
UNITS_SORTED = sorted(UNITS, key=len, reverse=True)
UNIT_PATTERN = "|".join(re.escape(u) for u in UNITS_SORTED)
QTY_PATTERN = (
    r"\d+\s*-\s*\d+"
    r"|\d+\s+\d+\s*/\s*\d+"
    r"|\d+\s*/\s*\d+"
    r"|\d+[.,]\d+"
    r"|\d+"
    r"|sesuai selera"
    r"|secukupnya"
    r"|sejumput"
    r"|beberapa"
)
# catatan: \b sengaja HANYA di belakang unit, bukan di depan -- supaya
# qty+unit yang nempel tanpa spasi (mis. '100gr') tetap kebaca, tapi unit
# tidak match sebagian kata di tengah nama bahan (mis. 'l' di 'lengkuas')
INGREDIENT_REGEX = re.compile(
    rf"^\s*(?P<qty>{QTY_PATTERN})?\s*(?:(?P<unit>{UNIT_PATTERN})\b)?\.?\s*(?P<name>.+?)\s*$",
    re.IGNORECASE,
)


def clean_name(name: str) -> str:
    name = name.lower().strip()
    name = re.sub(r"\(.*?\)", "", name)
    name = re.sub(r"\s+", " ", name).strip()
    return name


def parse_ingredient(raw: str) -> dict:
    raw = html.unescape(raw).strip()
    raw = re.sub(r"^[\-\*•]+\s*", "", raw)
    main, note = (raw.split(",", 1) + [""])[:2] if "," in raw else (raw, "")
    main = main.strip()
    m = INGREDIENT_REGEX.match(main)
    if m and m.group("name"):
        name = clean_name(m.group("name"))
    else:
        name = clean_name(main)
    return {"raw": raw, "name": name}


def parse_user_ingredients(text: str) -> list:
    """Input: '2 butir telur, 100gr dada ayam, 2 siung bawang putih'
    Note: user memisahkan tiap bahan dgn koma, TAPI parse_ingredient juga memakai
    koma untuk memisahkan catatan (mis. 'dikocok lepas'). Untuk input user yang
    berupa daftar bahan bersih (tanpa catatan cara masak), kita split dulu per
    kemungkinan bahan pakai heuristik: pisah di koma yang diikuti oleh angka/qty
    word atau di awal token baru. Untuk kasus sederhana (tanpa catatan), split
    koma biasa sudah cukup."""
    items = [t.strip() for t in text.split(",") if t.strip()]
    parsed = [parse_ingredient(item) for item in items]
    return [p["name"] for p in parsed if p["name"]]


# test
user_input = "2 butir telur, 100gr dada ayam, 2 siung bawang putih"
user_ingredients = parse_user_ingredients(user_input)
print(user_ingredients)

['telur', 'dada ayam', 'bawang putih']


## 3. Skor Efisiensi Bahan

Definisi yang dipakai di sini:

- **matched** = jumlah bahan resep yang cocok dengan salah satu bahan user (matching berbasis substring dua arah, bukan exact match, supaya `"telur"` tetap match dengan `"telur ayam"`)
- **coverage_recipe** = matched / total bahan resep -> makin tinggi, makin sedikit bahan tambahan yang perlu dibeli user (ini inti "efisiensi")
- **coverage_user** = matched / total bahan yang diketik user -> makin tinggi, makin banyak bahan user yang benar-benar terpakai (bukan resep yang random cuma nyenggol 1 bahan)
- **efficiency_score** = harmonic mean dari `coverage_recipe` dan `coverage_user` (mirip F1-score) -> menyeimbangkan dua sisi tadi, dan resep yang timpang di salah satu sisi (misal cocok bahan tapi resepnya beda arah) akan dihukum

> Ini rule-based/baseline. Kalau di evaluasi hasilnya kurang bagus, opsi lanjutannya: matching pakai embedding similarity antar nama bahan (bukan cuma substring), atau pakai stemming/lemmatization Bahasa Indonesia (mis. Sastrawi).

In [5]:
def ingredient_matches(user_name: str, recipe_name: str) -> bool:
    u = user_name.lower().strip()
    r = recipe_name.lower().strip()
    if not u or not r:
        return False
    return u in r or r in u


def compute_efficiency_score(user_ingredients: list, recipe_ingredients: list) -> dict:
    if not recipe_ingredients:
        return {
            "matched": 0,
            "coverage_recipe": 0.0,
            "coverage_user": 0.0,
            "efficiency_score": 0.0,
            "matched_items": [],
            "missing_items": [],
        }

    matched_recipe_items = []
    matched_user_items = set()
    for r_name in recipe_ingredients:
        hit = False
        for u_name in user_ingredients:
            if ingredient_matches(u_name, r_name):
                hit = True
                matched_user_items.add(u_name)
        if hit:
            matched_recipe_items.append(r_name)

    matched = len(matched_recipe_items)
    missing_items = [r for r in recipe_ingredients if r not in matched_recipe_items]

    coverage_recipe = matched / len(recipe_ingredients)
    coverage_user = (
        len(matched_user_items) / len(user_ingredients) if user_ingredients else 0.0
    )

    if coverage_recipe + coverage_user == 0:
        efficiency_score = 0.0
    else:
        efficiency_score = (2 * coverage_recipe * coverage_user) / (
            coverage_recipe + coverage_user
        )

    return {
        "matched": matched,
        "coverage_recipe": coverage_recipe,
        "coverage_user": coverage_user,
        "efficiency_score": efficiency_score,
        "matched_items": matched_recipe_items,
        "missing_items": missing_items,
    }


# test cepat
recipe_ex = ["telur ayam", "dada ayam fillet", "bawang putih", "kecap manis", "garam"]
print(compute_efficiency_score(user_ingredients, recipe_ex))

{'matched': 3, 'coverage_recipe': 0.6, 'coverage_user': 1.0, 'efficiency_score': 0.7499999999999999, 'matched_items': ['telur ayam', 'dada ayam fillet', 'bawang putih'], 'missing_items': ['kecap manis', 'garam']}


## 4. Fungsi Reranking Utuh

Alur: retrieve `top_k_candidates` dari ChromaDB (lebih banyak dari yang akan ditampilkan) -> hitung efficiency score tiap kandidat -> gabung dengan sbert_score (dinormalisasi 0-1 dgn min-max di antara kandidat) -> urutkan berdasar `final_score` -> potong ke `top_n_final`.

In [6]:
def retrieve_candidates(user_text: str, top_k: int = 30) -> list:
    query_embedding = model.encode(
        [add_query_prefix(user_text)], normalize_embeddings=True, convert_to_numpy=True
    )
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )

    candidates = []
    for i in range(len(results["ids"][0])):
        meta = results["metadatas"][0][i]
        candidates.append(
            {
                "id": results["ids"][0][i],
                "title": meta["title"],
                "ingredient_names": json.loads(meta["ingredient_names"]),
                "steps_text": meta["steps_text"],
                "sbert_score": 1
                - results["distances"][0][i],  # cosine distance -> similarity
            }
        )
    return candidates


def rerank(
    user_text: str,
    top_k_candidates: int = 30,
    top_n_final: int = 5,
    w_sbert: float = 0.5,
    w_efficiency: float = 0.5,
) -> list:
    user_ingredients = parse_user_ingredients(user_text)
    candidates = retrieve_candidates(user_text, top_k=top_k_candidates)

    sbert_scores = np.array([c["sbert_score"] for c in candidates])
    s_min, s_max = sbert_scores.min(), sbert_scores.max()
    sbert_norm = (sbert_scores - s_min) / (s_max - s_min + 1e-9)

    for c, s_norm in zip(candidates, sbert_norm):
        eff = compute_efficiency_score(user_ingredients, c["ingredient_names"])
        c.update(eff)
        c["sbert_score_norm"] = float(s_norm)
        c["final_score"] = (
            w_sbert * float(s_norm) + w_efficiency * eff["efficiency_score"]
        )

    candidates_sorted = sorted(candidates, key=lambda c: c["final_score"], reverse=True)
    return candidates_sorted[:top_n_final], user_ingredients

## 5. Uji Coba & Bandingkan dengan Retrieval Murni

In [7]:
user_input = "2 butir telur, 100gr dada ayam, 2 siung bawang putih"

final_results, parsed_user_ingredients = rerank(
    user_input, top_k_candidates=30, top_n_final=5, w_sbert=0.5, w_efficiency=0.5
)

print("Bahan user (setelah parsing):", parsed_user_ingredients)
print("=" * 70)

for i, c in enumerate(final_results):
    print(f"[{i + 1}] {c['title']}")
    print(
        f"    final_score={c['final_score']:.4f}  "
        f"(sbert_norm={c['sbert_score_norm']:.3f}, efficiency={c['efficiency_score']:.3f})"
    )
    print(f"    bahan resep: {c['ingredient_names']}")
    print(f"    -> sudah dimiliki user: {c['matched_items']}")
    print(f"    -> perlu ditambah: {c['missing_items']}")
    print()

Bahan user (setelah parsing): ['telur', 'dada ayam', 'bawang putih']
[1] Telur Dadar Daun Bawang
    final_score=0.6250  (sbert_norm=1.000, efficiency=0.250)
    bahan resep: ['telur ayam', 'daun bawang merah', 'cabai', 'garam', 'merica bubuk']
    -> sudah dimiliki user: ['telur ayam']
    -> perlu ditambah: ['daun bawang merah', 'cabai', 'garam', 'merica bubuk']

[2] Telur Dadar
    final_score=0.5026  (sbert_norm=0.561, efficiency=0.444)
    bahan resep: ['telur', 'bawang merah', 'bawang putih', 'daun bawang', 'garam dan penyedap', 'minyak goreng']
    -> sudah dimiliki user: ['telur', 'bawang putih']
    -> perlu ditambah: ['bawang merah', 'daun bawang', 'garam dan penyedap', 'minyak goreng']

[3] Telur Dadar
    final_score=0.4880  (sbert_norm=0.576, efficiency=0.400)
    bahan resep: ['telur ayam', 'kornet sapi', 'bawang daun', 'keju cheddar', 'bawang putih bubuk', 'lada bubuk', 'garam']
    -> sudah dimiliki user: ['telur ayam', 'bawang putih bubuk']
    -> perlu ditambah: ['kor

## Catatan

- `w_sbert` dan `w_efficiency` adalah hyperparameter — coba beberapa kombinasi (mis. 0.7/0.3, 0.3/0.7) dan bandingkan hasilnya secara kualitatif, atau kalau punya ground truth/labeled data, bisa dievaluasi kuantitatif (mis. NDCG, precision@k).
- `top_k_candidates` (jumlah kandidat sebelum rerank) juga berpengaruh: makin besar, makin besar peluang resep yang "lengkap bahannya" tapi skor SBERT-nya biasa-biasa saja ikut terangkat -- tapi juga makin lambat.
- Field `missing_items` di setiap hasil ini persis yang kamu butuh untuk fitur *"apakah ada bahan pelengkap lainnya?"* -- itu daftar bahan yang belum dikonfirmasi user, bisa langsung dilempar ke prompt Llama3 di tahap berikutnya.

## 6. Hybrid Retrieval: SBERT + Keyword Fallback

**Masalah:** untuk query pendek (cuma 2-3 bahan), retrieval SBERT murni bisa saja tidak pernah memasukkan resep yang benar-benar match ke dalam top-K kandidat sama sekali -- walau resep itu ada di dataset. Ini karena embedding semantik cenderung mengelompokkan berdasarkan pola judul/topik dominan (mis. "telur dadar"), bukan menjamin kelengkapan bahan.

**Solusi:** gabungkan dua sumber kandidat sebelum reranking:
1. **SBERT candidates** -- hasil `collection.query()` seperti biasa (menangkap kemiripan semantik/topik)
2. **Keyword candidates** -- resep yang bahannya match langsung (substring) dengan input user, diambil dari seluruh koleksi (bukan cuma top-K SBERT) -- ini menjamin resep yang benar-benar relevan tidak akan terlewat gara-gara skor SBERT-nya kebetulan rendah

Kandidat dari keyword search yang skor SBERT-nya belum ada (karena tidak ikut ke-retrieve oleh ChromaDB) akan dihitung on-the-fly dengan encode ulang lalu cosine similarity terhadap query embedding, supaya tetap bisa dibandingkan adil dengan kandidat dari SBERT.

In [8]:
# cache seluruh koleksi di memori sekali saja (5000 dokumen, ringan)
_ALL_DOCS_CACHE = None


def get_all_documents():
    global _ALL_DOCS_CACHE
    if _ALL_DOCS_CACHE is None:
        raw = collection.get(include=["metadatas", "documents", "embeddings"])
        _ALL_DOCS_CACHE = list(
            zip(raw["ids"], raw["metadatas"], raw["documents"], raw["embeddings"])
        )
    return _ALL_DOCS_CACHE


def keyword_candidates(
    user_ingredients: list, exclude_ids: set, max_candidates: int = 20
) -> list:
    """Cari resep di SELURUH koleksi yang bahannya match keyword dgn input user,
    di luar id yang sudah ada di hasil SBERT.

    Diurutkan berdasarkan (matched_count, coverage_recipe) -- keduanya menurun.
    PENTING: tie-breaker coverage_recipe ditambahkan karena versi awal (cuma sort
    by matched_count) membiarkan urutan asli dataset jadi tie-breaker implisit,
    yang bisa memotong kandidat yang sebenarnya lebih efisien hanya karena posisinya
    di dataset kebetulan lebih belakang. Lihat kasus 'Ayam kukus telur' (coverage=0.5,
    posisi ke-23 dari urutan lama) yang kepotong dari batas 20 kandidat sebelum fix ini."""
    scored = []
    for doc_id, meta, document, embedding in get_all_documents():
        if doc_id in exclude_ids:
            continue
        names = json.loads(meta["ingredient_names"])
        matched = sum(
            1 for u in user_ingredients if any(ingredient_matches(u, n) for n in names)
        )
        if matched > 0:
            coverage = matched / len(names) if names else 0.0
            scored.append((matched, coverage, doc_id, meta, document, embedding))

    scored.sort(key=lambda x: (-x[0], -x[1]))
    return [
        (m, doc_id, meta, document, embedding)
        for m, cov, doc_id, meta, document, embedding in scored[:max_candidates]
    ]


def retrieve_candidates_hybrid(
    user_text: str,
    user_ingredients: list,
    top_k_sbert: int = 30,
    max_keyword_extra: int = 20,
) -> list:
    query_embedding = model.encode(
        [add_query_prefix(user_text)], normalize_embeddings=True, convert_to_numpy=True
    )[0]

    # 1) kandidat dari SBERT (seperti sebelumnya)
    sbert_candidates = retrieve_candidates(user_text, top_k=top_k_sbert)
    existing_ids = {c["id"] for c in sbert_candidates}

    # 2) kandidat tambahan dari keyword search di seluruh koleksi
    kw_extra = keyword_candidates(
        user_ingredients, exclude_ids=existing_ids, max_candidates=max_keyword_extra
    )

    for matched, doc_id, meta, document, embedding in kw_extra:
        sbert_score = float(
            np.dot(query_embedding, np.array(embedding))
        )  # cosine, krn sudah dinormalisasi
        sbert_candidates.append(
            {
                "id": doc_id,
                "title": meta["title"],
                "ingredient_names": json.loads(meta["ingredient_names"]),
                "steps_text": meta["steps_text"],
                "sbert_score": sbert_score,
            }
        )

    return sbert_candidates


def rerank_hybrid(
    user_text: str,
    top_k_sbert: int = 30,
    max_keyword_extra: int = 20,
    top_n_final: int = 5,
    w_sbert: float = 0.5,
    w_efficiency: float = 0.5,
) -> tuple:
    user_ingredients = parse_user_ingredients(user_text)
    candidates = retrieve_candidates_hybrid(
        user_text, user_ingredients, top_k_sbert, max_keyword_extra
    )

    sbert_scores = np.array([c["sbert_score"] for c in candidates])
    s_min, s_max = sbert_scores.min(), sbert_scores.max()
    sbert_norm = (sbert_scores - s_min) / (s_max - s_min + 1e-9)

    for c, s_norm in zip(candidates, sbert_norm):
        eff = compute_efficiency_score(user_ingredients, c["ingredient_names"])
        c.update(eff)
        c["sbert_score_norm"] = float(s_norm)
        c["final_score"] = (
            w_sbert * float(s_norm) + w_efficiency * eff["efficiency_score"]
        )

    candidates_sorted = sorted(candidates, key=lambda c: c["final_score"], reverse=True)
    return candidates_sorted[:top_n_final], user_ingredients, len(candidates)

In [9]:
user_input = "2 butir telur, 100gr dada ayam, 2 siung bawang putih"

final_results, parsed_user_ingredients, n_pool = rerank_hybrid(
    user_input,
    top_k_sbert=30,
    max_keyword_extra=20,
    top_n_final=5,
    w_sbert=0.5,
    w_efficiency=0.5,
)

print("Bahan user (setelah parsing):", parsed_user_ingredients)
print("Total kandidat yang dibandingkan (SBERT + keyword fallback):", n_pool)
print("=" * 70)

for i, c in enumerate(final_results):
    print(f"[{i + 1}] {c['title']}")
    print(
        f"    final_score={c['final_score']:.4f}  "
        f"(sbert_norm={c['sbert_score_norm']:.3f}, efficiency={c['efficiency_score']:.3f})"
    )
    print(f"    bahan resep: {c['ingredient_names']}")
    print(f"    -> sudah dimiliki user: {c['matched_items']}")
    print(f"    -> perlu ditambah: {c['missing_items']}")
    print()

Bahan user (setelah parsing): ['telur', 'dada ayam', 'bawang putih']
Total kandidat yang dibandingkan (SBERT + keyword fallback): 50
[1] Telur Dadar
    final_score=0.6554  (sbert_norm=0.866, efficiency=0.444)
    bahan resep: ['telur', 'bawang merah', 'bawang putih', 'daun bawang', 'garam dan penyedap', 'minyak goreng']
    -> sudah dimiliki user: ['telur', 'bawang putih']
    -> perlu ditambah: ['bawang merah', 'daun bawang', 'garam dan penyedap', 'minyak goreng']

[2] Telur Dadar
    final_score=0.6355  (sbert_norm=0.871, efficiency=0.400)
    bahan resep: ['telur ayam', 'kornet sapi', 'bawang daun', 'keju cheddar', 'bawang putih bubuk', 'lada bubuk', 'garam']
    -> sudah dimiliki user: ['telur ayam', 'bawang putih bubuk']
    -> perlu ditambah: ['kornet sapi', 'bawang daun', 'keju cheddar', 'lada bubuk', 'garam']

[3] Telur Dadar Daun Bawang
    final_score=0.6250  (sbert_norm=1.000, efficiency=0.250)
    bahan resep: ['telur ayam', 'daun bawang merah', 'cabai', 'garam', 'merica b

## 7. Perbandingan 3 Skenario Formula Efisiensi

Membandingkan 3 pendekatan pada **kandidat yang sama** (hasil hybrid retrieval), supaya perbandingannya adil:

- **Skenario A -- Harmonic mean (baseline)**: `efficiency = 2*coverage_recipe*coverage_user / (coverage_recipe+coverage_user)`, bobot final `0.5 SBERT : 0.5 efisiensi`. Ini yang sudah dipakai sejauh ini -- adil ke `coverage_recipe` dan `coverage_user`, tapi resep dengan banyak bahan total gampang dirugikan walau semua bahan user terpakai.
- **Skenario B -- F-beta (beta=2, condong ke coverage_user)**: `F_beta = (1+beta^2) * coverage_recipe*coverage_user / (beta^2*coverage_recipe + coverage_user)`. Dengan `beta=2`, `coverage_user` (seberapa banyak bahan yg diketik user benar-benar terpakai) dibobot 4x lebih penting daripada `coverage_recipe`. Bobot final tetap `0.5 : 0.5`.
- **Skenario C -- Bobot digeser ke efisiensi**: formula tetap harmonic mean seperti skenario A, tapi bobot final digeser jadi `0.3 SBERT : 0.7 efisiensi` -- efisiensi bahan lebih dominan dibanding kemiripan semantik.

> `beta=2` dan bobot `0.3:0.7` di sini masih pilihan awal -- kalau hasil skenario B/C kelihatan menjanjikan, nilai ini juga bisa di-tuning lebih lanjut.

In [10]:
def coverage_stats(user_ingredients: list, recipe_ingredients: list):
    """Hitung coverage_recipe, coverage_user, matched_items, missing_items -- dipakai bersama oleh semua skenario."""
    if not recipe_ingredients:
        return 0.0, 0.0, [], []
    matched_recipe_items, matched_user_items = [], set()
    for r_name in recipe_ingredients:
        hit = False
        for u_name in user_ingredients:
            if ingredient_matches(u_name, r_name):
                hit = True
                matched_user_items.add(u_name)
        if hit:
            matched_recipe_items.append(r_name)
    missing_items = [r for r in recipe_ingredients if r not in matched_recipe_items]
    coverage_recipe = len(matched_recipe_items) / len(recipe_ingredients)
    coverage_user = (
        len(matched_user_items) / len(user_ingredients) if user_ingredients else 0.0
    )
    return coverage_recipe, coverage_user, matched_recipe_items, missing_items


def harmonic_efficiency(coverage_recipe: float, coverage_user: float) -> float:
    if coverage_recipe + coverage_user == 0:
        return 0.0
    return (2 * coverage_recipe * coverage_user) / (coverage_recipe + coverage_user)


def fbeta_efficiency(
    coverage_recipe: float, coverage_user: float, beta: float = 2.0
) -> float:
    """beta > 1 -> coverage_user (recall thd bahan user) dibobot lebih penting drpd coverage_recipe (precision)."""
    denom = (beta**2 * coverage_recipe) + coverage_user
    if denom == 0:
        return 0.0
    return (1 + beta**2) * (coverage_recipe * coverage_user) / denom

In [11]:
def rerank_compare(
    user_text: str,
    top_k_sbert: int = 30,
    max_keyword_extra: int = 20,
    top_n_final: int = 5,
    beta: float = 2.0,
):
    user_ingredients = parse_user_ingredients(user_text)
    candidates = retrieve_candidates_hybrid(
        user_text, user_ingredients, top_k_sbert, max_keyword_extra
    )

    sbert_scores = np.array([c["sbert_score"] for c in candidates])
    s_min, s_max = sbert_scores.min(), sbert_scores.max()
    sbert_norm = (sbert_scores - s_min) / (s_max - s_min + 1e-9)

    for c, s_norm in zip(candidates, sbert_norm):
        cr, cu, matched_items, missing_items = coverage_stats(
            user_ingredients, c["ingredient_names"]
        )
        eff_harmonic = harmonic_efficiency(cr, cu)
        eff_fbeta = fbeta_efficiency(cr, cu, beta=beta)

        c["sbert_score_norm"] = float(s_norm)
        c["coverage_recipe"] = cr
        c["coverage_user"] = cu
        c["matched_items"] = matched_items
        c["missing_items"] = missing_items
        c["eff_harmonic"] = eff_harmonic
        c["eff_fbeta"] = eff_fbeta

        c["final_A_harmonic_5050"] = 0.5 * s_norm + 0.5 * eff_harmonic
        c["final_B_fbeta_5050"] = 0.5 * s_norm + 0.5 * eff_fbeta
        c["final_C_harmonic_3070"] = 0.3 * s_norm + 0.7 * eff_harmonic

    scenarios = {
        "A: Harmonic mean, bobot 0.5 SBERT : 0.5 efisiensi (baseline)": "final_A_harmonic_5050",
        f"B: F-beta (beta={beta}, condong coverage_user), bobot 0.5 : 0.5": "final_B_fbeta_5050",
        "C: Harmonic mean, bobot 0.3 SBERT : 0.7 efisiensi": "final_C_harmonic_3070",
    }

    results = {}
    for label, key in scenarios.items():
        results[label] = sorted(candidates, key=lambda c: c[key], reverse=True)[
            :top_n_final
        ]

    return results, user_ingredients, len(candidates)


def print_scenario_results(label: str, results: list):
    # deteksi key skor otomatis dari label (lebih robust drpd hardcode map)
    if label.startswith("A:"):
        score_key = "final_A_harmonic_5050"
    elif label.startswith("B:"):
        score_key = "final_B_fbeta_5050"
    elif label.startswith("C:"):
        score_key = "final_C_harmonic_3070"
    else:
        score_key = "final_D_fbeta_3070"
    print(f"\n{'=' * 80}\n{label}\n{'=' * 80}")
    for i, c in enumerate(results):
        print(
            f"[{i + 1}] {c['title']}  (final={c[score_key]:.3f}, sbert_norm={c['sbert_score_norm']:.3f}, "
            f"eff_harmonic={c['eff_harmonic']:.3f}, eff_fbeta={c['eff_fbeta']:.3f})"
        )
        print(
            f"    bahan resep ({len(c['ingredient_names'])}): {c['ingredient_names']}"
        )
        print(f"    -> sudah dimiliki user: {c['matched_items']}")
        print(f"    -> perlu ditambah: {c['missing_items']}")

In [12]:
user_input = "2 butir telur, 100gr dada ayam, 2 siung bawang putih"

results, parsed_user_ingredients, n_pool = rerank_compare(
    user_input, top_k_sbert=30, max_keyword_extra=20, top_n_final=5, beta=2.0
)

print("Bahan user (setelah parsing):", parsed_user_ingredients)
print("Total kandidat yang dibandingkan:", n_pool)

for label, res in results.items():
    print_scenario_results(label, res)

Bahan user (setelah parsing): ['telur', 'dada ayam', 'bawang putih']
Total kandidat yang dibandingkan: 50

A: Harmonic mean, bobot 0.5 SBERT : 0.5 efisiensi (baseline)
[1] Telur Dadar  (final=0.655, sbert_norm=0.866, eff_harmonic=0.444, eff_fbeta=0.556)
    bahan resep (6): ['telur', 'bawang merah', 'bawang putih', 'daun bawang', 'garam dan penyedap', 'minyak goreng']
    -> sudah dimiliki user: ['telur', 'bawang putih']
    -> perlu ditambah: ['bawang merah', 'daun bawang', 'garam dan penyedap', 'minyak goreng']
[2] Telur Dadar  (final=0.636, sbert_norm=0.871, eff_harmonic=0.400, eff_fbeta=0.526)
    bahan resep (7): ['telur ayam', 'kornet sapi', 'bawang daun', 'keju cheddar', 'bawang putih bubuk', 'lada bubuk', 'garam']
    -> sudah dimiliki user: ['telur ayam', 'bawang putih bubuk']
    -> perlu ditambah: ['kornet sapi', 'bawang daun', 'keju cheddar', 'lada bubuk', 'garam']
[3] Telur Dadar Daun Bawang  (final=0.625, sbert_norm=1.000, eff_harmonic=0.250, eff_fbeta=0.294)
    bahan re

## 8. Guardrail: Cegah Llama3 "Mengarang" Resep

Dua kondisi yang HARUS ditangani sebelum data sampai ke Llama3 -- keduanya rule-based, bukan keputusan LLM:

1. **Input cuma bumbu/pelengkap, tanpa bahan utama** (mis. "garam, gula, kecap") -- tidak ada gunanya di-retrieve, langsung minta bahan utama.
2. **Kombinasi bahan tidak nyambung** (mis. "ayam, gula") -- retrieval tetap jalan tapi skor kandidat terbaiknya akan sangat rendah. Kalau di bawah ambang, JANGAN tampilkan hasilnya seolah-olah itu rekomendasi valid.

**Prinsip desain untuk tahap Llama3 nanti:** Llama3 HANYA boleh menyampaikan ulang (dalam bahasa natural) resep yang sudah ada di `results` -- termasuk kalimat penolakan/permintaan klarifikasi di bawah ini sebaiknya juga template tetap yang di-paraphrase Llama3, BUKAN Llama3 yang memutuskan sendiri kapan harus menolak. Llama3 tidak pernah diberi izin untuk membuat resep sendiri dari nol.

In [13]:
# Daftar bahan yang TERGOLONG bumbu/pelengkap murni (heuristik, bukan daftar sempurna --
# sesuaikan/lengkapi berdasarkan hasil testing, mis. jika ada bahan yang salah terklasifikasi)
BUMBU_ONLY = {
    "garam",
    "gula",
    "gula pasir",
    "gula merah",
    "gula jawa",
    "gula aren",
    "merica",
    "merica bubuk",
    "lada",
    "lada bubuk",
    "ketumbar",
    "ketumbar bubuk",
    "kunyit",
    "kunyit bubuk",
    "jahe",
    "lengkuas",
    "kemiri",
    "terasi",
    "kecap manis",
    "kecap asin",
    "kecap inggris",
    "saus tiram",
    "saus sambal",
    "saus tomat",
    "saus asam manis",
    "minyak goreng",
    "minyak wijen",
    "penyedap rasa",
    "penyedap",
    "kaldu bubuk",
    "kaldu ayam bubuk",
    "kaldu sapi bubuk",
    "daun salam",
    "daun jeruk",
    "daun jeruk purut",
    "serai",
    "sereh",
    "vanili",
    "baking powder",
    "baking soda",
    "ragi",
    "pewarna makanan",
    "msg",
    "micin",
    "cuka",
    "air asam jawa",
    "asam jawa",
    "garam dan penyedap",
}


def classify_ingredient(name: str) -> str:
    """'bumbu' kalau match persis dgn daftar BUMBU_ONLY, selain itu dianggap
    'utama_atau_lainnya' -- default-nya permisif (benefit of the doubt) supaya bahan
    yang tidak ada di daftar tidak salah ditolak."""
    name = name.lower().strip()
    if name in BUMBU_ONLY:
        return "bumbu"
    return "utama_atau_lainnya"


def has_main_ingredient(user_ingredients: list) -> bool:
    """True kalau minimal ada 1 bahan yg BUKAN bumbu murni."""
    return any(classify_ingredient(u) != "bumbu" for u in user_ingredients)


# test
print(
    has_main_ingredient(["garam", "gula", "kecap manis"])
)  # False -> minta bahan utama
print(has_main_ingredient(["telur", "garam"]))  # True -> lanjut retrieval
print(has_main_ingredient(["ayam", "gula"]))  # True -> lanjut, tapi cek relevansi nanti

False
True
True


In [14]:
def is_relevant_enough(best_candidate: dict, user_ingredients: list) -> bool:
    """T_relevansi = min(2, jumlah bahan user) -- menyesuaikan panjang input:
    - user kasih 1 bahan -> wajib match bahan itu (T=1)
    - user kasih >=2 bahan -> wajib match minimal 2 bahan (T=2), bukan cuma 1.
    Ini mencegah threshold tetap (mis. selalu >=2) salah menolak query pendek yang valid,
    sekaligus mencegah threshold longgar (>=1) meloloskan kombinasi yang sebenarnya
    tidak related (lihat kasus 'soda kue, kunyit' vs 'ayam, gula')."""
    if not best_candidate or not user_ingredients:
        return False
    T_relevansi = min(2, len(user_ingredients))
    return len(best_candidate.get("matched_items", [])) >= T_relevansi


def handle_user_input(user_text: str, top_n: int = 5) -> dict:
    """Pipeline utuh dgn guardrail. Return dict dgn 'status' yg menentukan respons apa
    yang boleh disampaikan Llama3 -- Llama3 TIDAK pernah membuat konten resep sendiri."""
    user_ingredients = parse_user_ingredients(user_text)

    if not user_ingredients:
        return {
            "status": "no_ingredients",
            "template": "Sepertinya belum ada bahan yang terdeteksi dari pesanmu. Boleh sebutkan bahan yang kamu punya?",
        }

    if not has_main_ingredient(user_ingredients):
        return {
            "status": "bumbu_only",
            "user_ingredients": user_ingredients,
            "template": (
                f"Bahan yang kamu sebutkan ({', '.join(user_ingredients)}) sepertinya baru bumbu/pelengkap. "
                f"Ada bahan utama juga nggak, misalnya ayam, telur, tahu, atau sayuran?"
            ),
        }

    results = rerank_hybrid(user_text, top_n_final=top_n)[0]

    if not is_relevant_enough(results[0] if results else None, user_ingredients):
        return {
            "status": "no_match",
            "user_ingredients": user_ingredients,
            "template": (
                f"Maaf, sistem belum menemukan resep yang cukup cocok untuk kombinasi bahan "
                f"({', '.join(user_ingredients)}). Coba sebutkan bahan lain atau kombinasi berbeda ya."
            ),
        }

    return {
        "status": "ok",
        "user_ingredients": user_ingredients,
        "results": results,
    }


# test ketiga kasus
for test_input in [
    "garam, gula, kecap manis",
    "ayam, gula",
    "2 butir telur, 100gr dada ayam, 2 siung bawang putih",
]:
    print(f"Input: {test_input!r}")
    result = handle_user_input(test_input)
    print(f"  status: {result['status']}")
    if result["status"] != "ok":
        print(f"  template: {result['template']}")
    else:
        print(f"  top-1: {result['results'][0]['title']}")
    print()

Input: 'garam, gula, kecap manis'
  status: bumbu_only
  template: Bahan yang kamu sebutkan (garam, gula, kecap manis) sepertinya baru bumbu/pelengkap. Ada bahan utama juga nggak, misalnya ayam, telur, tahu, atau sayuran?

Input: 'ayam, gula'
  status: ok
  top-1: Ayam gulai bumbu instan Racik Indofood

Input: '2 butir telur, 100gr dada ayam, 2 siung bawang putih'
  status: ok
  top-1: Telur Dadar



## 9. Grid Search Bobot Final (w1, w2)

Menjalankan grid search sesungguhnya menggunakan 30 query uji (perluas ke sini sesuai jumlah final kamu) untuk menentukan nilai $w_1$ dan $w_2$ yang dilaporkan sebagai final di BAB IV. Bobot dipilih berdasarkan rata-rata IAR@5 tertinggi di seluruh query uji.

**Jalankan cell ini setelah semua cell dari notebook 03 di atas sudah dieksekusi** (butuh `model`, `collection`, `retrieve_candidates_hybrid`, `coverage_stats`, dkk. dari cell-cell sebelumnya).

In [15]:
test_queries_full = [
    "2 butir telur, 100gr dada ayam, 2 siung bawang putih",
    "200gr daging sapi, kecap manis, bawang bombay",
    "tahu, tempe, cabai rawit",
    "udang, tepung terigu, telur",
    "wortel, kentang, buncis",
    "tempe, kacang panjang, cabai merah",
    "2 butir telur",
    "nasi sisa kemarin, 1 butir telur",
    "1 papan tempe, kecap manis",
    "dada ayam fillet dikit",
    "indomie goreng, sawi hijau, cabe rawit",
    "roti tawar, keju cheddar",
    "1 buah tahu, 1 buah tomat, daun bawang",
    "500 gr daging sapi, 1 buah bawang bombay, kecap inggris",
    "1 ikat bayam, jagung manis, bawang merah",
    "kangkung, terasi, cabai rawit, bawang merah",
    "ikan nila, bumbu kuning, kemangi",
    "sosis sapi, nasi putih, bawang bombay, saus sambal",
    "200 gr udang, brokoli, bawang putih, saus tiram",
    "ikan lele, jeruk nipis, garam, ketumbar",
    "500gr ayam, santan kental, daun salam, serai, lengkuas, bawang merah, bawang putih",
    "daging sapi, kelapa sangrai, cabai merah besar, bawang merah, bawang putih, jahe, lengkuas",
    "nasi putih, telur, sosis, bakso, kecap manis, saus tomat, sayur kol, daun bawang",
    "dada ayam, tepung bumbu, telur, bawang putih, saus asam manis, nanas, paprika",
    "tahu, tempe, kacang panjang, tauge, bumbu kacang, kerupuk",
    "tepung terigu, mentega, gula pasir, telur, vanili",
    "pisang kepok, tepung beras, gula merah, minyak goreng",
    "bubuk matcha murni, susu cair, es batu",
    "250 gr jamur enoki, gochujang, minyak wijen",
    "frozen kentang goreng, sosis sapi, bumbu tabur jagung bakar",
]
# 30 query -- sama persis dengan yang dipakai di notebook 04 (evaluasi otomatis awal),
# supaya bobot final konsisten diuji pada set query yang sama dengan yang dilaporkan di BAB I.


def evaluate_weight(w1: float, w2: float, top_n: int = 5) -> dict:
    """Rata-rata IAR@top_n dan SBERT@top_n, dengan skor_efisiensi_bahan = harmonic_efficiency
    (coverage_recipe, coverage_user), BUKAN cuma coverage_recipe. Ini mencegah resep pendek
    yang cuma nyerempet 1 bahan user menang cuma karena total bahannya sedikit (lihat temuan
    'Ayam kukus telur' vs 'Dadar telur 1 Menit' di sub-bab spot-check)."""
    iars, sberts = [], []
    for q in test_queries_full:
        user_ingredients = parse_user_ingredients(q)
        candidates = retrieve_candidates_hybrid(
            q, user_ingredients, top_k_sbert=30, max_keyword_extra=20
        )
        sbert_scores = np.array([c["sbert_score"] for c in candidates])
        s_min, s_max = sbert_scores.min(), sbert_scores.max()
        sbert_norm = (sbert_scores - s_min) / (s_max - s_min + 1e-9)
        for c, s_norm in zip(candidates, sbert_norm):
            cr, cu, matched, missing = coverage_stats(
                user_ingredients, c["ingredient_names"]
            )
            c["coverage_recipe"] = cr
            c["coverage_user"] = cu
            c["skor_efisiensi_bahan"] = harmonic_efficiency(cr, cu)
            c["sbert_score_norm"] = float(s_norm)
            c["final_score"] = w1 * float(s_norm) + w2 * c["skor_efisiensi_bahan"]
        top = sorted(candidates, key=lambda c: c["final_score"], reverse=True)[:top_n]
        # IAR yang dilaporkan tetap coverage_recipe (definisi IAR asli, sub-bab 2.2.7/3.1)
        iars.append(np.mean([c["coverage_recipe"] for c in top]))
        sberts.append(np.mean([c["sbert_score_norm"] for c in top]))
    return {"mean_iar": float(np.mean(iars)), "mean_sbert": float(np.mean(sberts))}


print(f"{'w1':>6} {'w2':>6} {'mean_IAR@5':>12} {'mean_SBERT@5':>14}")
results = []
for w1 in np.arange(
    0.30, 0.51, 0.05
):  # grid diperhalus di rentang 0.30-0.50 (increment 0.05)
    w2 = round(1 - w1, 2)
    w1 = round(w1, 2)
    metrics = evaluate_weight(w1, w2)
    results.append((w1, w2, metrics["mean_iar"], metrics["mean_sbert"]))
    print(
        f"{w1:>6} {w2:>6} {metrics['mean_iar']:>12.4f} {metrics['mean_sbert']:>14.4f}"
    )

print(
    "\nCatatan: dengan skor_efisiensi_bahan = harmonic_efficiency(coverage_recipe, coverage_user),"
)
print(
    "pola trade-off di atas TETAP perlu dibaca dengan hati-hati (coverage_recipe masih salah"
)
print(
    "satu komponennya), tapi resep yang cuma nyerempet 1 bahan tanpa memanfaatkan bahan user"
)
print(
    "lainnya sekarang akan mendapat penalti lewat komponen coverage_user yang rendah."
)

    w1     w2   mean_IAR@5   mean_SBERT@5
   0.3    0.7       0.3593         0.7899
  0.35   0.65       0.3478         0.8187
   0.4    0.6       0.3401         0.8321
  0.45   0.55       0.3268         0.8482
   0.5    0.5       0.3177         0.8599

Catatan: dengan skor_efisiensi_bahan = harmonic_efficiency(coverage_recipe, coverage_user),
pola trade-off di atas TETAP perlu dibaca dengan hati-hati (coverage_recipe masih salah
satu komponennya), tapi resep yang cuma nyerempet 1 bahan tanpa memanfaatkan bahan user
lainnya sekarang akan mendapat penalti lewat komponen coverage_user yang rendah.


## 10. Spot-Check Kualitatif dengan Bobot Final (w1=0.4, w2=0.6)

Validasi manual: lihat langsung top-5 hasil di beberapa query pakai bobot yang direkomendasikan dari analisis trade-off (sub-bab grid search di atas). Tujuannya mastiin hasilnya masuk akal secara kualitatif, bukan cuma bagus di angka agregat.

In [16]:
W1_FINAL = 0.6  # hasil elbow-point analysis dari grid search 30 query (final)
W2_FINAL = 0.4


def rerank_with_weights(
    user_text: str,
    w1: float,
    w2: float,
    top_n: int = 5,
    top_k_sbert: int = 30,
    max_keyword_extra: int = 20,
) -> tuple:
    user_ingredients = parse_user_ingredients(user_text)
    candidates = retrieve_candidates_hybrid(
        user_text, user_ingredients, top_k_sbert, max_keyword_extra
    )

    sbert_scores = np.array([c["sbert_score"] for c in candidates])
    s_min, s_max = sbert_scores.min(), sbert_scores.max()
    sbert_norm = (sbert_scores - s_min) / (s_max - s_min + 1e-9)

    for c, s_norm in zip(candidates, sbert_norm):
        cr, cu, matched, missing = coverage_stats(
            user_ingredients, c["ingredient_names"]
        )
        c["coverage_recipe"] = cr
        c["coverage_user"] = cu
        c["skor_efisiensi_bahan"] = harmonic_efficiency(cr, cu)
        c["sbert_score_norm"] = float(s_norm)
        c["matched_items"] = matched
        c["missing_items"] = missing
        c["final_score"] = w1 * float(s_norm) + w2 * c["skor_efisiensi_bahan"]

    top = sorted(candidates, key=lambda c: c["final_score"], reverse=True)[:top_n]
    return top, user_ingredients


spot_check_queries = [
    "2 butir telur, 100gr dada ayam, 2 siung bawang putih",
    "tahu, tempe, cabai rawit",
    "ikan lele, jeruk nipis, garam, ketumbar",
    "daging sapi, kecap manis, bawang bombay",
]

for q in spot_check_queries:
    top, user_ingredients = rerank_with_weights(q, W1_FINAL, W2_FINAL)
    print(f"Query: {q!r}")
    print(f"Bahan user (setelah parsing): {user_ingredients}")
    print("-" * 70)
    for i, c in enumerate(top):
        print(
            f"[{i + 1}] {c['title']}  (final={c['final_score']:.3f}, "
            f"sbert_norm={c['sbert_score_norm']:.3f}, "
            f"cov_recipe={c['coverage_recipe']:.3f}, cov_user={c['coverage_user']:.3f}, "
            f"efisiensi={c['skor_efisiensi_bahan']:.3f})"
        )
        print(
            f"    bahan resep ({len(c['ingredient_names'])}): {c['ingredient_names']}"
        )
        print(f"    -> sudah dimiliki user: {c['matched_items']}")
        print(f"    -> perlu ditambah: {c['missing_items']}")
    print("=" * 70)
    print()

print("CHECKLIST VALIDASI MANUAL (isi berdasarkan hasil di atas):")
print("[ ] Apakah top-1 di tiap query terlihat masuk akal secara tema/topik masakan?")
print(
    "[ ] Apakah 'bahan yang perlu ditambah' terlihat wajar (bumbu dasar), bukan bahan aneh/tidak nyambung?"
)
print(
    "[ ] Apakah resep dgn cov_user rendah (cuma nyerempet 1 bahan) sudah TIDAK menang lagi"
)
print("    dibanding resep dengan cov_user tinggi (pakai semua bahan user)?")
print(
    "Kalau ada hasil yang janggal, catat contohnya untuk didiskusikan sebagai limitasi di BAB IV."
)

Query: '2 butir telur, 100gr dada ayam, 2 siung bawang putih'
Bahan user (setelah parsing): ['telur', 'dada ayam', 'bawang putih']
----------------------------------------------------------------------
[1] Telur Dadar Daun Bawang  (final=0.700, sbert_norm=1.000, cov_recipe=0.200, cov_user=0.333, efisiensi=0.250)
    bahan resep (5): ['telur ayam', 'daun bawang merah', 'cabai', 'garam', 'merica bubuk']
    -> sudah dimiliki user: ['telur ayam']
    -> perlu ditambah: ['daun bawang merah', 'cabai', 'garam', 'merica bubuk']
[2] Telur Dadar  (final=0.698, sbert_norm=0.866, cov_recipe=0.333, cov_user=0.667, efisiensi=0.444)
    bahan resep (6): ['telur', 'bawang merah', 'bawang putih', 'daun bawang', 'garam dan penyedap', 'minyak goreng']
    -> sudah dimiliki user: ['telur', 'bawang putih']
    -> perlu ditambah: ['bawang merah', 'daun bawang', 'garam dan penyedap', 'minyak goreng']
[3] Telur Dadar  (final=0.683, sbert_norm=0.871, cov_recipe=0.286, cov_user=0.667, efisiensi=0.400)
    baha